# Snapshot 6 May Results Plots

This notebook:

1. Collects results from the fast 1-fold eval tree.
2. Builds merged tables for `tree`, `sbm`, and `pa`.
3. Produces one plot for each dataset with a shared legend outside the axes.


In [ ]:
from __future__ import annotations

import json
import re
import shutil
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd

REPO = Path('/mnt/lts4/scratch/home/mmonteir/jacobi_diff/jacobi-graph-diffusion')
RESULTS_ROOT = REPO / 'samples' / 'drive_graphs_snapshot_6may' / 'evals_fast_1fold'

DATASET_CONFIG = {
    'tree': {
        'metric': 'forest_acc',
        'x_range': (20, 200),
        'in_dist': (20, 80),
        'title': r'Tree',
        'ylabel': r'Forest acc.',
    },
    'sbm': {
        'metric': 'sbm_acc',
        'x_range': (40, 200),
        'in_dist': (40, 80),
        'title': r'SBM',
        'ylabel': r'SBM acc.',
    },
    'pa': {
        'metric': 'pa_acc',
        'x_range': (40, 300),
        'in_dist': (40, 80),
        'title': r'PA',
        'ylabel': r'PA acc.',
    },
}

METHOD_ORDER = ['diphon', 'defog', 'digress', 'gdss', 'grum']
METHOD_LABELS = {
    'defog': 'DeFoG',
    'digress': 'DiGress',
    'diphon': 'DiPhon',
    'gdss': 'GDSS',
    'grum': 'GRUM',
}
USE_TEX = shutil.which('latex') is not None

mpl.rcParams.update({
    'text.usetex': USE_TEX,
    'font.family': 'serif',
    'font.serif': ['Times', 'Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset': 'cm',
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.02,
    'axes.linewidth': 0.6,
    'axes.labelsize': 8,
    'axes.titlesize': 8,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'legend.fontsize': 7,
    'lines.linewidth': 1.2,
    'lines.markersize': 3.8,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.major.size': 2.5,
    'ytick.major.size': 2.5,
    'xtick.major.width': 0.6,
    'ytick.major.width': 0.6,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})
if USE_TEX:
    mpl.rcParams['text.latex.preamble'] = r'\usepackage{amsmath}'

METHOD_STYLES = {
    'defog': dict(color='#0072B2', marker='o', linestyle='-'),
    'digress': dict(color='#D55E00', marker='s', linestyle='--'),
    'diphon': dict(color='#009E73', marker='D', linestyle='-.'),
    'gdss': dict(color='#CC79A7', marker='^', linestyle=':'),
    'grum': dict(color='#E69F00', marker='P', linestyle='-'),
}


In [ ]:
def parse_size_from_stem(stem: str):
    patterns = [
        r'_n(\d+)(?:_|$)',
        r'_(\d+)_gdss$',
        r'_(\d+)_grum$',
        r'pa_diphon_(\d+)$',
        r'sbm_diphon_(\d+)$',
        r'_sample_n(\d+)$',
        r'^n(\d+)_g\d+_jacobi_input_nx_graphs$',
    ]
    for pattern in patterns:
        m = re.search(pattern, stem)
        if m:
            return int(m.group(1))
    return None


def extract_metric_from_record(dataset: str, metric_name: str, payload: dict):
    if 'metrics_mean_std' in payload:
        block = payload['metrics_mean_std']
        if metric_name in block:
            value = block[metric_name]
            if isinstance(value, dict):
                return value.get('mean')
            return value

    if 'accuracy' in payload:
        if dataset in {'sbm', 'pa'} and metric_name in {'sbm_acc', 'pa_acc'}:
            return payload['accuracy']
        if dataset == 'tree' and metric_name == 'tree_acc':
            return payload['accuracy']

    if dataset == 'tree' and metric_name == 'forest_acc':
        return payload.get('acc_extra', {}).get('forest_acc')

    return None


def extract_records_from_metrics_json(path: Path, dataset: str, source_kind: str):
    with open(path, 'r') as f:
        payload = json.load(f)

    method = path.parent.parent.name
    stem = path.parent.name
    metric_name = DATASET_CONFIG[dataset]['metric']
    records = []

    if isinstance(payload, dict) and 'entries' in payload:
        for entry in payload['entries']:
            size = entry.get('size')
            metric_value = extract_metric_from_record(dataset, metric_name, entry)
            if size is None or metric_value is None:
                continue
            records.append({
                'dataset': dataset,
                'method': method,
                'size': int(size),
                'metric_name': metric_name,
                'metric_value': float(metric_value),
                'source': source_kind,
                'path': str(path),
            })
        return records

    size = payload.get('size')
    if isinstance(size, list):
        size = None
    if size is None:
        size = parse_size_from_stem(stem)

    metric_value = extract_metric_from_record(dataset, metric_name, payload)
    if size is None or metric_value is None:
        return records

    records.append({
        'dataset': dataset,
        'method': method,
        'size': int(size),
        'metric_name': metric_name,
        'metric_value': float(metric_value),
        'source': source_kind,
        'path': str(path),
    })
    return records


def collect_records(root: Path, source_kind: str):
    rows = []
    for dataset in DATASET_CONFIG:
        dataset_root = root / dataset
        if not dataset_root.exists():
            continue
        for metrics_path in sorted(dataset_root.glob('*/*/metrics.json')):
            rows.extend(extract_records_from_metrics_json(metrics_path, dataset, source_kind))
    return pd.DataFrame(rows)


merged_df = (
    collect_records(RESULTS_ROOT, 'fast')
    .sort_values(['dataset', 'method', 'size'])
    .reset_index(drop=True)
)

merged_df.head()


In [ ]:
def dataset_table(dataset: str):
    cfg = DATASET_CONFIG[dataset]
    x_min, x_max = cfg['x_range']
    sub = merged_df[(merged_df['dataset'] == dataset) & (merged_df['size'].between(x_min, x_max))].copy()
    sub['method'] = pd.Categorical(sub['method'], categories=METHOD_ORDER, ordered=True)
    table = (
        sub.pivot_table(index='size', columns='method', values='metric_value', aggfunc='first', observed=False)
        .sort_index()
        .reindex(columns=[m for m in METHOD_ORDER if m in sub['method'].unique()])
    )
    return table


tables = {dataset: dataset_table(dataset) for dataset in DATASET_CONFIG}

for dataset, table in tables.items():
    print(f'\n=== {dataset.upper()} ===')
    display(table)


In [ ]:
FIGURE_DIR = REPO / 'analysis' / 'figures' / 'accuracy_results'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.05), sharey=True)
plot_method_order = ['diphon', 'defog', 'digress', 'gdss', 'grum']
legend_handles = {}
panel_labels = (
    {'tree': r'\textbf{(a)} Tree', 'sbm': r'\textbf{(b)} SBM', 'pa': r'\textbf{(c)} PA'}
    if USE_TEX
    else {'tree': '(a) Tree', 'sbm': '(b) SBM', 'pa': '(c) PA'}
)

for ax, dataset in zip(axes, ['tree', 'sbm', 'pa']):
    cfg = DATASET_CONFIG[dataset]
    x_min, x_max = cfg['x_range']
    in_min, in_max = cfg['in_dist']
    metric_name = cfg['metric']
    sub = merged_df[(merged_df['dataset'] == dataset) & (merged_df['size'].between(x_min, x_max))].copy()

    ax.axvspan(in_min, in_max, color='0.92', zorder=0)
    ax.axvline(in_min, color='0.68', linewidth=0.6, linestyle='--', zorder=1)
    ax.axvline(in_max, color='0.68', linewidth=0.6, linestyle='--', zorder=1)

    for method in plot_method_order:
        method_df = sub[sub['method'] == method].sort_values('size')
        if method_df.empty:
            continue
        style = METHOD_STYLES[method]
        (line,) = ax.plot(
            method_df['size'],
            method_df['metric_value'],
            label=METHOD_LABELS.get(method, method),
            markeredgewidth=0.4,
            markeredgecolor='white',
            **style,
        )
        legend_handles[method] = line

    ax.set_title(panel_labels[dataset], loc='center', pad=3)
    ax.set_xlabel(r'Number of nodes')
    ax.set_ylabel(r'Accuracy ($\uparrow$)')
    ax.set_xlim(x_min, x_max)
    ax.set_xticks(range(x_min, x_max + 1, 20))
    ax.tick_params(axis='x', labelrotation=35)
    ax.set_ylim(0.0, 1.02)
    ax.grid(axis='y', color='0.88', linewidth=0.5)
    ax.set_axisbelow(True)
    ax.margins(x=0.01)

for ax in axes[1:]:
    ax.set_ylabel('')

handles = [legend_handles[m] for m in plot_method_order if m in legend_handles]
labels = [METHOD_LABELS[m] for m in plot_method_order if m in legend_handles]
fig.legend(
    handles,
    labels,
    loc='upper center',
    bbox_to_anchor=(0.5, 1.10),
    ncol=len(labels),
    frameon=False,
    handlelength=1.8,
    columnspacing=1.1,
    handletextpad=0.4,
)
fig.tight_layout(pad=0.25, w_pad=0.7)

pdf_path = FIGURE_DIR / 'snapshot6_accuracy_neurips.pdf'
png_path = FIGURE_DIR / 'snapshot6_accuracy_neurips.png'
fig.savefig(pdf_path)
fig.savefig(png_path)
plt.show()


In [ ]:
failure_fig_dir = FIGURE_DIR
failure_fig_dir.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.05), sharey=True)
plot_method_order = ['diphon', 'defog', 'digress', 'gdss', 'grum']
legend_handles = {}
panel_labels = (
    {'tree': r'\textbf{(a)} Tree', 'sbm': r'\textbf{(b)} SBM', 'pa': r'\textbf{(c)} PA'}
    if USE_TEX
    else {'tree': '(a) Tree', 'sbm': '(b) SBM', 'pa': '(c) PA'}
)

for ax, dataset in zip(axes, ['tree', 'sbm', 'pa']):
    cfg = DATASET_CONFIG[dataset]
    x_min, x_max = cfg['x_range']
    in_min, in_max = cfg['in_dist']
    sub = merged_df[(merged_df['dataset'] == dataset) & (merged_df['size'].between(x_min, x_max))].copy()

    ax.axvspan(in_min, in_max, color='0.92', zorder=0)
    ax.axvline(in_min, color='0.68', linewidth=0.6, linestyle='--', zorder=1)
    ax.axvline(in_max, color='0.68', linewidth=0.6, linestyle='--', zorder=1)

    for method in plot_method_order:
        method_df = sub[sub['method'] == method].sort_values('size')
        if method_df.empty:
            continue
        style = METHOD_STYLES[method]
        failure_rate = 100.0 * (1.0 - method_df['metric_value'])
        (line,) = ax.plot(
            method_df['size'],
            failure_rate,
            label=METHOD_LABELS.get(method, method),
            markeredgewidth=0.4,
            markeredgecolor='white',
            **style,
        )
        legend_handles[method] = line

    ax.set_title(panel_labels[dataset], loc='center', pad=3)
    ax.set_xlabel(r'Number of nodes')
    ax.set_ylabel(r'Failure rate ($\%$) $\downarrow$')
    ax.set_xlim(x_min, x_max)
    ax.set_xticks(range(x_min, x_max + 1, 20))
    ax.tick_params(axis='x', labelrotation=35)
    ax.set_ylim(bottom=0.0)
    ax.grid(axis='y', color='0.88', linewidth=0.5)
    ax.set_axisbelow(True)
    ax.margins(x=0.01)

for ax in axes[1:]:
    ax.set_ylabel('')

handles = [legend_handles[m] for m in plot_method_order if m in legend_handles]
labels = [METHOD_LABELS[m] for m in plot_method_order if m in legend_handles]
fig.legend(
    handles,
    labels,
    loc='upper center',
    bbox_to_anchor=(0.5, 1.10),
    ncol=len(labels),
    frameon=False,
    handlelength=1.8,
    columnspacing=1.1,
    handletextpad=0.4,
)
fig.tight_layout(pad=0.25, w_pad=0.7)

failure_pdf_path = failure_fig_dir / 'snapshot6_failure_rate_neurips.pdf'
failure_png_path = failure_fig_dir / 'snapshot6_failure_rate_neurips.png'
fig.savefig(failure_pdf_path)
fig.savefig(failure_png_path)
plt.show()
